# Log-Gaussian Cox process conditioned on observed events

A log-Gaussian Cox process (LGCP) is an inhomogeneous Poisson process whose intensity $\lambda(t) = e^{f(t)}$ has a GP prior on the log-intensity $f$:

$$
f \sim \mathcal{GP}(\mu, k), \qquad \{t_i\} \mid f \sim \text{PoissonProcess}\bigl(e^{f}\bigr), \qquad
\log p(\{t_i\} \mid f) = \sum_{i} f(t_i) - \int_0^T e^{f(t)}\, \mathrm{d}t.
$$

This notebook samples from the GP prior with a flow and uses guidance with this log-likelihood to steer the samples towards the posterior $p(f \mid \{t_i\})$ given simulated event times. It compares two discretisations of the likelihood: counting events per grid bin, and interpolating $f$ at the exact event times.

## 1. Setup

`T` is the observation window and `N` the size of the inference grid; the ground truth is simulated on a finer grid of `N_TRUTH` points. `TRUE_*` are the data-generating GP parameters, which are reused for the prior; in practice they would be estimated from the data.

In [ ]:
import gpytorch
import matplotlib.pyplot as plt
import numpy as np
import torch

from flow_inference.conditions import BaseCondition
from flow_inference.flow_utils import build_grid_in_box
from flow_inference.flows import GaussianProcessFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.sampler import IntegratorSampler

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 1066 if device == "cuda" else 42  # chosen for a clear visual example
T = 10.0            # observation window [0, T]
N = 256             # inference grid points
N_TRUTH = 2000      # fine grid for simulating the ground truth
NUM_SAMPLES = 100
NUM_STEPS = 500

TRUE_LENGTHSCALE = 1.0
TRUE_OUTPUTSCALE = 10.0
TRUE_MEAN = 1.0

## 2. Simulated data and GP prior

### 2.1 Ground truth

Draw the true log-intensity $f^*$ from the GP on a fine grid.

In [ ]:
torch.manual_seed(SEED)
t_truth = torch.linspace(0.0, T, N_TRUTH, device=device)

kernel_true = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel()).to(device)
kernel_true.base_kernel.lengthscale = TRUE_LENGTHSCALE
kernel_true.outputscale = TRUE_OUTPUTSCALE

# Drawing the true GP path needs a Cholesky of its kernel matrix, which is too badly
# conditioned for float32, so this one-off simulation runs in float64.
with torch.no_grad():
    t64 = t_truth.double().unsqueeze(-1)
    K_truth = kernel_true.double()(t64).to_dense()
    L_truth = torch.linalg.cholesky(K_truth + 1e-6 * torch.eye(N_TRUTH, device=device, dtype=torch.float64))
    f_truth = (TRUE_MEAN + L_truth @ torch.randn(N_TRUTH, device=device, dtype=torch.float64)).float()
    kernel_true.float()
lambda_truth = torch.exp(f_truth)

### 2.2 Simulated events

Simulate events by Lewis–Shedler thinning: propose candidates from a homogeneous Poisson process at rate $\lambda_{\max}$, and accept each candidate $t$ with probability $\lambda^*(t)/\lambda_{\max}$, with $\lambda^*$ linearly interpolated from the fine grid.

In [ ]:
lambda_max = lambda_truth.max().item()
dt_truth = T / N_TRUTH
n_candidates = torch.poisson(torch.tensor(lambda_max * T)).long().item()
t_candidates = torch.rand(n_candidates, device=device) * T

idx_lo = (t_candidates / dt_truth).long().clamp(0, N_TRUTH - 2)
frac = (t_candidates / dt_truth) - idx_lo.float()
lam_at_candidates = (1 - frac) * lambda_truth[idx_lo] + frac * lambda_truth[idx_lo + 1]

accept = torch.rand(n_candidates, device=device) < lam_at_candidates / lambda_max
t_events = t_candidates[accept].sort().values

print(f"Expected events: {(lambda_truth.mean() * T).item():.1f}, observed: {len(t_events)}")

The ground truth: the log-intensity $f^*$ and the intensity $\lambda^* = e^{f^*}$, with the simulated events.

In [ ]:
t_np = t_truth.cpu().numpy()
f_np = f_truth.cpu().numpy()
lam_np = lambda_truth.cpu().numpy()
ev_np = t_events.cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(t_np, f_np, color="steelblue", lw=1.5, label="$f^*(t)$")
axes[0].axhline(TRUE_MEAN, color="gray", lw=0.8, ls="--", label="mean")
axes[0].set_ylabel("log-intensity $f(t)$")
axes[0].legend(frameon=False)

axes[1].plot(t_np, lam_np, color="darkorange", lw=1.5, label="$\\lambda^*(t) = e^{f^*(t)}$")
axes[1].scatter(ev_np, np.zeros(len(ev_np)), marker="|", color="black",
                s=60, linewidths=0.8, label=f"events ($n={len(ev_np)}$)")
axes[1].set_xlabel("$t$")
axes[1].set_ylabel("intensity $\\lambda(t)$")
axes[1].legend(frameon=False)

fig.suptitle("Log-Gaussian Cox process — ground truth", fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 GP prior

Both likelihoods share the same prior flow on an $N$-point grid. `build_grid_in_box` returns coordinates in $[0, 1)$, so the lengthscale is divided by $T$, and `t_infer` maps the grid back to the original time units for plotting.

In [ ]:
coords = build_grid_in_box(N, 1, device)  # (N, 1)

kernel_prior = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel()).to(device)
kernel_prior.base_kernel.lengthscale = TRUE_LENGTHSCALE / T
kernel_prior.outputscale = TRUE_OUTPUTSCALE

prior_mean = gpytorch.means.ConstantMean().to(device)
with torch.no_grad():
    prior_mean.constant.fill_(TRUE_MEAN)

flow = GaussianProcessFlow(kernel_prior, prior_mean, coords, device=device)
sampler = IntegratorSampler(flow=flow)

t_infer = coords.squeeze(-1).cpu().numpy() * T  # grid in original time units

## 3. Poisson likelihoods

Both likelihoods are full log-likelihoods rather than residuals with a tolerance, so each subclasses `BaseCondition` directly and has no $\sigma$ to choose.

### 3.1 Binned likelihood

Bin the events into the $N$ grid cells. Cell $j$ has count $n_j$ and Poisson mean $e^{f_j}\Delta t$, so

$$
\log p(\{n_j\} \mid f) = \sum_{j} \bigl[ n_j f_j - e^{f_j} \Delta t \bigr]
$$

up to a constant. With $N = 256$, the bin width $\Delta t = T/N \approx 0.04$ is much shorter than the lengthscale of $f$, so the binning error is small.

In [ ]:
class BinnedPoissonCondition(BaseCondition):
    def __init__(self, event_counts: torch.Tensor, T: float):
        super().__init__()
        self.register_buffer("n", event_counts)
        self.register_buffer("dt", torch.tensor(T / len(event_counts)))

    def logp(self, x: torch.Tensor) -> torch.Tensor:
        f = x.squeeze(1)
        return (self.n * f - self.dt * torch.exp(f)).sum(dim=-1)

### 3.2 Exact likelihood (interpolated)

Evaluate $f$ at each event time $t_i$ by linear interpolation of the grid values, and approximate $\int_0^T e^{f(t)}\,\mathrm{d}t$ by a Riemann sum on the grid. The interpolation weights are precomputed, so `logp` is a cheap gather.

In [ ]:
class ExactPoissonCondition(BaseCondition):
    def __init__(self, event_times: torch.Tensor, T: float, N: int):
        super().__init__()
        self.register_buffer("dt", torch.tensor(T / N))

        # Linear interpolation weights of each event onto the grid
        frac_idx = (event_times / T * N).clamp(0.0, N - 1.0 - 1e-9)
        idx_lo = frac_idx.long()
        frac = frac_idx - idx_lo.to(frac_idx.dtype)
        self.register_buffer("idx_lo", idx_lo)
        self.register_buffer("w_lo", 1.0 - frac)
        self.register_buffer("w_hi", frac)

    def logp(self, x: torch.Tensor) -> torch.Tensor:
        f = x.squeeze(1)  # (B, N)
        f_at_events = self.w_lo * f[:, self.idx_lo] + self.w_hi * f[:, self.idx_lo + 1]
        event_term = f_at_events.sum(dim=-1)
        integral = self.dt * torch.exp(f).sum(dim=-1)
        return event_term - integral

## 4. Conditioned sampling

`plot_posterior` summarises a batch of samples by their pointwise mean and 90% interval, for both $f$ and the intensity $e^f$, against the truth.

In both scenarios below, `GuidanceMC` estimates the guidance term at each step from one Monte Carlo draw of the clean function given the current state, and the sampler uses the `"snr"` time schedule, which spaces the steps evenly in log signal-to-noise ratio. The seed is reset before each run, so the two differ only in the likelihood.

In [ ]:
def plot_posterior(samples, title=""):
    """Posterior mean and 90% interval of f and exp(f), against the truth."""
    samples_f = samples.squeeze(1).cpu().numpy()  # (B, N)
    post_mean = samples_f.mean(axis=0)
    post_lo = np.quantile(samples_f, 0.05, axis=0)
    post_hi = np.quantile(samples_f, 0.95, axis=0)

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(t_np, f_np, "k-", lw=2, label="$f^*$ (truth)", zorder=4)
    axes[0].fill_between(t_infer, post_lo, post_hi, color="steelblue", alpha=0.3, label="90% CI")
    axes[0].plot(t_infer, post_mean, color="steelblue", lw=1.5, label="posterior mean")
    axes[0].set_ylabel("log-intensity $f(t)$")
    axes[0].legend(frameon=False)

    axes[1].plot(t_np, lam_np, "k-", lw=2, label="$\\lambda^*$ (truth)", zorder=4)
    axes[1].fill_between(t_infer, np.exp(post_lo), np.exp(post_hi), color="darkorange", alpha=0.3, label="90% CI")
    axes[1].plot(t_infer, np.exp(post_mean), color="darkorange", lw=1.5, label="posterior mean")
    axes[1].scatter(ev_np, np.zeros(len(ev_np)), marker="|", color="black",
                    s=60, linewidths=0.8, label=f"events ($n={len(ev_np)}$)")
    axes[1].set_xlabel("$t$")
    axes[1].set_ylabel("intensity $\\lambda(t)$")
    axes[1].legend(frameon=False)

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

### 4.1 Binned likelihood

Count the events in each grid cell, then sample with the binned condition.

In [ ]:
bin_indices = (t_events / T * N).long().clamp(0, N - 1)
event_counts = torch.zeros(N, device=device)
event_counts.scatter_add_(0, bin_indices, torch.ones(len(bin_indices), device=device))

condition = BinnedPoissonCondition(event_counts, T).to(device)
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)

torch.manual_seed(SEED)
samples = sampler.sample(NUM_SAMPLES, guidance=guidance, num_steps=NUM_STEPS, time_schedule="snr")
plot_posterior(samples, title="Binned Poisson likelihood, GuidanceMC")

### 4.2 Exact likelihood (interpolated)

The same, with the condition built from the exact event times.

In [ ]:
condition = ExactPoissonCondition(t_events, T, N).to(device)
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)

torch.manual_seed(SEED)
samples = sampler.sample(NUM_SAMPLES, guidance=guidance, num_steps=NUM_STEPS, time_schedule="snr")
plot_posterior(samples, title="Exact Poisson likelihood (interpolated), GuidanceMC")

## 5. Summary

* A point-process likelihood is a `BaseCondition` whose `logp` is the log-likelihood on the grid, with the event data precomputed into buffers.
* Guidance turns samples from the GP prior into approximate posterior samples of the log-intensity, given only the event times.
* The two discretisations should give very similar posteriors here, because the bins are much narrower than the lengthscale of $f$; compare the two figures, and check whether the 90% interval covers the true intensity.